# 8. Learned reference-ratio histograms with 10M selected events per source

Generate fresh independent simulation **in Colab**, applying notebook 02's
frozen preselection until each of S, SBI, B, NI, NI-up and NI-down has at least
10 million selected events. Use exactly 10 million per source for the primary
histograms, with 250k and 1M prefixes as MC convergence controls.
Keep the filtered trained ensembles, no floor, power $a=100$ and 36 primary bins.
Generation, GPU predictions and histogram accumulation resume from Drive.

### Runtime and persistent run

Use **a Colab GPU runtime** and Run all from the top. Existing models and the
preselection are reused; no training or cluster submission is needed. The new
simulation is independent of the original training and integration samples.
Keep RUN_NAME, Unity model directory and filter policy as in notebook 03.
The Unity directory here is only the existing model bundle already on Drive.

Checkpoints persist on Drive; interrupted chunks are replayed. Restart the
runtime and Run all to resume. GPU model files are staged on Colab local disk;
events and predictions are processed in bounded chunks. The 60M selected events
are never concatenated in RAM. Allow roughly 2.2 GB on Drive for selected
coordinates and three float64 ratios per event, plus small manifests/templates;
local model staging uses additional disk. Rejected events are not saved.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

SOURCE_COMMIT = "c567cfaee8c8cf6b70ae747646c2b7c3f89e7cc6"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    import tempfile
    # Download afresh even in a reused runtime; do not alter older checkouts.
    REPO_DIR = Path(tempfile.mkdtemp(prefix="poodemo-08-", dir="/content"))
    print(f"Notebook 08 source: {SOURCE_COMMIT} (fresh download)")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import urllib.error
        import urllib.request

        archive_url = f"https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/{SOURCE_COMMIT}"

        def fetch_source(token=None):
            headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
            if token:
                headers["Authorization"] = "Bearer " + token
            request = urllib.request.Request(archive_url, headers=headers)
            with urllib.request.urlopen(request, timeout=180) as response:
                return response.read()

        try:
            archive_bytes = fetch_source()
        except urllib.error.HTTPError as error:
            if error.code not in (401, 403, 404):
                raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if not token:
                token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
            if not token:
                raise RuntimeError("Provide a GitHub token with read access to download the pinned notebook 08 source.")
            try:
                archive_bytes = fetch_source(token)
            except urllib.error.HTTPError as auth_error:
                raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
            finally:
                token = None

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                for entry in archive.getmembers():
                    safe_destination(entry.name)
                    if not (entry.isfile() or entry.isdir()):
                        raise RuntimeError("Links and special files are not supported in the source archive.")
                archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The downloaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    # Run from the top: subsequent cells recreate objects from this checkout.
    # Clear all poodemo submodules, including old plotting and fit functions.
    for module_name in list(sys.modules):
        if module_name == "poodemo" or module_name.startswith("poodemo."):
            del sys.modules[module_name]

else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
# Observable-only settings: keep these identical in notebooks 08 and 09.
# The floor changes the observable, never the physical mu or its fit bounds.
N_BINS = 36
REFERENCE_RATIO_ETA_MIN = 0.0  # No floor.
REFERENCE_RATIO_A = 100.0
if not np.isfinite(REFERENCE_RATIO_A) or REFERENCE_RATIO_A <= 0:
    raise ValueError("REFERENCE_RATIO_A must be finite and positive")
if not np.isfinite(REFERENCE_RATIO_ETA_MIN) or not 0 <= REFERENCE_RATIO_ETA_MIN < 1:
    raise ValueError("REFERENCE_RATIO_ETA_MIN must lie in [0, 1)")
run = run.with_config(
    reference_ratio_power=float(REFERENCE_RATIO_A),
    reference_ratio_eta_floor=float(REFERENCE_RATIO_ETA_MIN),
)
print(f"Reference ratio: {N_BINS} bins, eta floor={REFERENCE_RATIO_ETA_MIN:g}, "
      f"a={REFERENCE_RATIO_A:g}; 0.5 is inside a bin")
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### Reuse notebook 03's filtered density ratios

Run notebook 03 with the transferred Unity models first. Keep the same
`RUN_NAME`, `MODE`, `CONFIG_OVERRIDES`, `UNITY_MODEL_DIR`, and
`ENSEMBLE_FILTER_POLICY` here. The default directory is
`My Drive/poodemo/unity_models/paper-distinct-s-v4-unity100/`; it contains
`bundle.json`, `campaign.json`, and `ratios/` directly inside it.

The next cell verifies the transferred bundle and reuses notebook 03's frozen
retained-member lists for all five ensembles, including the pointwise outlier
rules. It verifies input, policy and artifact hashes without re-screening or
evaluating analytical densities. Changed inputs require a completed matching
filter run in notebook 03 first; no networks are trained here. `RATIO_SOURCE = "drive"` explicitly
selects the original Drive models instead, with the same filtering rules.

The next cells create an independent Colab simulation cache. The selected
filtered models remain unchanged; predictions on the new events are cached.
All new results use a separate study directory printed below.

In [ ]:
import os
from pathlib import Path
from poodemo.data import save_json
from poodemo.ensemble_bundle import use_bundle, BundleEvaluationRun
from poodemo.member_filter import MemberFilterPolicy, unfiltered_run
from poodemo.practical import (PracticalRun, use_practical_run,
                               reuse_filtered_ensembles as filter_ratio_ensembles)

RATIO_SOURCE = "unity"  # "drive" explicitly selects the original saved models.
UNITY_MODEL_DIR = Path(os.environ.get(
    "POODEMO_UNITY_MODEL_DIR",
    "/content/drive/MyDrive/poodemo/unity_models/paper-distinct-s-v4-unity100",
))
ENSEMBLE_FILTER_POLICY = MemberFilterPolicy(
    catastrophic_peer_factor=100.0,
)  # Match notebook 03, including any custom policy settings.

# Unwrap a previous selection so rerunning this cell never nests evaluations.
source_run = run.base_run if isinstance(run, PracticalRun) else run
source_run = unfiltered_run(source_run)
source_run = source_run.base_run if isinstance(source_run, BundleEvaluationRun) else source_run
if RATIO_SOURCE == "unity":
    if not (UNITY_MODEL_DIR / "bundle.json").is_file():
        raise FileNotFoundError(
            f"Copy the complete Unity model bundle to {UNITY_MODEL_DIR}. "
            "Expected bundle.json, campaign.json and ratios/ directly inside it. "
            "Use the same directory as notebook 03; see unity/README.md.")
    ensemble_run = use_bundle(source_run, UNITY_MODEL_DIR)
elif RATIO_SOURCE == "drive":
    ensemble_run = source_run
else:
    raise ValueError("RATIO_SOURCE must be 'drive' or 'unity'")
run = filter_ratio_ensembles(ensemble_run, policy=ENSEMBLE_FILTER_POLICY)
filter_summary = pd.read_csv(run.path("results", "ensemble_filter", "summary.csv"))
display(filter_summary)
print("Ratio source:", RATIO_SOURCE)
print("Retained ratio models:", run.path("models", "ratios"))
print("Evaluation outputs:", run.evaluation_root)
save_json(run.path("results", "reference_ratio_model_source.json"), {
    "ratio_source": RATIO_SOURCE,
    "bundle_directory": str(ensemble_run.bundle) if RATIO_SOURCE == "unity" else None,
    "campaign_id": ensemble_run.bundle_manifest["campaign_id"] if RATIO_SOURCE == "unity" else None,
    "filter_identity": run.filter_manifest["identity"],
    "filter_policy": run.filter_manifest["inputs"]["policy"],
    "evaluation_root": str(run.evaluation_root),
})
_filtered_run = run
run = use_practical_run(run)
print("Practical evaluation:", run.evaluation_root)

### What changes with the larger sample

Fresh events determine the component **shapes**. The selected process yields
$\nu_j$ remain the estimates from the original integration samples, and the
learned-map normalizers remain fixed on the original held-out S sample.
Each prefix of $N$ fresh selected events receives weight $w_j=\nu_j/N$.
Increasing MC statistics does not change the experimental luminosity or yields.
Acceptance-rate uncertainty is not reduced by this shape-only convergence study.
The selected-target stopping count is never used as an acceptance estimate.

All coordinates use learned ratios. The only analytical inference calculation
is the unbinned benchmark; analytical PDFs are not histogram weights.
Same-template Asimov observations test each fixed MC model, not independent
closure against the true experiment. Prefixes are nested and correlated.
NI retains the existing after-binning morph, including its explicit sparse-bin
convention. Negative physical predictions and nonregular Fisher points remain
invalid; no yield floor or automatic merging is introduced.

### Generate and evaluate fresh selected events

The target is **per source after preselection**, not a cap on existing samples.
Progress and hashes are saved after each completed chunk and each ratio task.
A new seed creates a separate independent campaign. Changing batch/chunk settings
creates a separate cache; keep them unchanged when resuming.

In [ ]:
from poodemo.colab_mc import (prepare_colab_samples, fill_colab_histograms,
                               template_diagnostics, convergence_scans)
from poodemo.pipeline import _positive_study_grid
import torch

SELECTED_EVENTS_PER_SOURCE = 10_000_000  # AFTER the frozen notebook-02 preselection.
MC_PREFIXES = [250_000, 1_000_000, SELECTED_EVENTS_PER_SOURCE]
MC_SEED = 20261111
SELECTED_CHUNK = 250_000
GENERATION_CHUNK = 250_000
INFERENCE_BATCH = 8192  # Reduce if the GPU runs out of memory.
NORMALIZATION_EVENTS = 250_000  # Original held-out S sample; fixed across MC prefixes.
BIN_COUNTS = [4, 12, 24, 36, 60]
ETA_VALUES = [0.0, 0.3, 0.6, 1.0, 1.5]
COMPARISON_ETA_MIN = 0.2
CHECK_ETAS = [0., 0.99, 0.999, 1., 1.001, 1.01]

if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun from the top.")
print("Inference device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU smoke check")
print(f"Target: {SELECTED_EVENTS_PER_SOURCE:,} selected events for EACH of S, SBI, B, NI, NI-up, NI-down")
print("Generation and neural evaluation run here in Colab; no cluster jobs or retraining.")

# Preserve the original practical run for calibration and sample caching.
mc_run = run
samples = prepare_colab_samples(
    mc_run, selected_per_source=SELECTED_EVENTS_PER_SOURCE, seed=MC_SEED,
    selected_chunk=SELECTED_CHUNK, generation_chunk=GENERATION_CHUNK,
    inference_batch=INFERENCE_BATCH, normalization_events=NORMALIZATION_EVENTS,
    model_cache=Path('/content/poodemo-model-cache') if IN_COLAB else None,
)
display(samples['sample_counts'])
print('Persistent sample cache:', samples['directory'])

In [ ]:
# NN predictions are cached. Changing eta/bin settings only refills histograms.
scan_grid = _positive_study_grid(mc_run)
requested_etas = np.unique(np.r_[scan_grid, ETA_VALUES, CHECK_ETAS, 1.,
                                 COMPARISON_ETA_MIN, REFERENCE_RATIO_ETA_MIN,
                                 np.maximum(scan_grid, REFERENCE_RATIO_ETA_MIN)])
requested_bins = sorted(set(BIN_COUNTS + [N_BINS]))
templates = fill_colab_histograms(
    mc_run, samples, etas=requested_etas, bin_counts=requested_bins,
    prefixes=MC_PREFIXES,
)
# Separate results protect historical 08/09 runs and different MC studies.
run = templates.study_run(mc_run)
print('10M-study outputs:', run.evaluation_root)
print('Stored direct templates:', templates.directory / 'counts.npz')

### Physical-domain and MC precision checks

For each bin, minimize the physical prediction exactly as a quadratic in
$\sqrt{\mu}$ over the configured fit bounds, at a declared grid of NI values.
The nuisance grid is a diagnostic, not a guarantee between its points. Save MC
uncertainties and zero-yield/nonzero-derivative checks at the generating truth.
These run before likelihood fits. Increasing the MC target does not guarantee
that every rare bin becomes valid.

In [ ]:
diagnostics = template_diagnostics(run, templates, etas=scan_grid, n_bins=N_BINS)
by_partition = diagnostics.groupby(['selected_per_source', 'eta']).agg(
    minimum_expected=('minimum_expected', 'min'),
    negative_bins=('negative', 'sum'),
    nonregular_bins_at_truth=('nonregular_mu_at_truth', 'sum'),
    incomplete_NI_support_bins=('incomplete_NI_support_bins', 'max'),
).reset_index()
display(by_partition.groupby('selected_per_source').agg(
    partitions=('eta', 'size'), minimum_expected=('minimum_expected', 'min'),
    partitions_with_negative_bins=('negative_bins', lambda x: int((x > 0).sum())),
    nonregular_partitions_at_truth=('nonregular_bins_at_truth', lambda x: int((x > 0).sum())),
))
problem_bins = diagnostics.loc[diagnostics.negative | diagnostics.nonregular_mu_at_truth]
if len(problem_bins):
    print('Unsupported physical predictions remain. No yields are clipped or floored.')
    display(problem_bins.head(30))
print('Full bin diagnostics, including MC uncertainties:',
      run.path('results', 'reference_ratio_mc_bin_diagnostics.csv'))

### Ratio to the fixed physical reference

At nominal NI nuisance, use the filtered learned **selected, normalized** ratios:
$$
\eta_{\mathrm{eff}}=\max(\eta,\eta_{\min}),\qquad
R_\eta(x)=\frac{r(x;\eta_{\mathrm{eff}})}{r(x;1)}
=\frac{p(x;\eta_{\mathrm{eff}},0)}{p(x;1,0)},
\qquad z_\eta(x)=\frac{R_\eta(x)^a}{1+R_\eta(x)^a}
=\operatorname{sigmoid}\!\left(a\log R_\eta(x)\right).
$$
Here $p$ denotes the learned model: $\hat r_S=1$ and
$\hat f_\eta(x)=\sum_j c_j(\eta)\nu_j\hat r_j(x)/Z_j$, with
$Z_j=\langle\hat r_j\rangle_{S,\mathrm{MC}}$ and selected MC rates $\nu_j$.
Then $\hat p_\eta/\hat p_1=(\hat f_\eta/\nu_\eta)/(\hat f_1/\nu_1)$;
the unknown common $S$ density cancels. No analytical PDF enters this map.
The rates and $Z_j$ are frozen once on the integration samples, never on data. The denominator is always at
$(\mu,\alpha_{NI})=(1,0)$, independent of the fit, anchor and Asimov truth settings.
The event count retains the Poisson rate information.

Use **36 bins**, **`REFERENCE_RATIO_ETA_MIN = 0.0`** and
**`REFERENCE_RATIO_A = 100`** in notebooks 08 and 09. The baseline has no floor:
$\eta_{\mathrm{eff}}=\eta$. Physical test points, generating truth, coefficients,
and fit bounds retain their original definitions. The power stays fixed inside
both likelihood fits. This notebook validates the new high-statistics direct templates first.
Notebook 09 is not automatically switched to the new cache by this update.

An exact ratio would preserve its fixed-NI endpoint-pair comparison; learned ratios need not.
Finite binning and NI profiling can still lose information, and pairwise
sufficiency does not imply sufficiency for every freely fitted physical $\mu$.
Compare to **analytical unbinned** as the sole oracle benchmark.

The former floor $0.2$ remains an explicit comparison setting. A positive floor
changes only the observable: below it, different physical hypotheses share a
partition but retain their own expected yields. Notebook 08 compares local
widths at fixed truth $\mu_*=1$; notebook 09 compares complete direct scans.
Neither comparison establishes confidence-interval coverage or global branch
stability, which require paired toys with the complete free fit.

At $\eta=1$, $R=1$ and $z=1/2$ exactly for every event. The separate equality
collapse remains: no jitter, shifted high-eta endpoint or score substitution is
introduced. That frozen histogram measures only the total rate. Near 1 the
distribution still contracts for any finite positive $a$. Larger $a$ spreads ratios
near 1 across more of $[0,1]$; $a=1$ gives the original $R/(1+R)$ transformation.

Every histogram uses `observable_bin_edges`: **0.5 is strictly inside a bin**.
Odd bin counts use uniform edges. For even counts, the internal edges shift
left by half a nominal bin width, so the central bins keep equal widths and
0.5 is a bin center (for at least four bins). Only the endpoint bins have widths
0.5 and 1.5 times the usual width. The count and endpoints 0 and 1 are preserved.
The same edges are used in plots, direct fits, estimator diagnostics and splines.
These are event-yield plots per bin, not densities divided by bin width.

### How the observable changes with eta

Each curve uses the observable constructed at $\eta$ itself in the no-floor
baseline. Edit `ETA_VALUES` and `N_BINS` below. $\eta=0$ is supported for these
distribution plots; likelihood and information scans use positive test points.
The separate exact collapse at $\eta=1$ remains.

In [ ]:
import mplhep as hep
from poodemo.pipeline import observable_bin_edges
OBSERVABLE = 'reference_ratio'
PLOT_PREFIX = '08'
OBSERVABLE_LABEL = rf"$z_\eta=R_\eta^a/(1+R_\eta^a),\quad a={REFERENCE_RATIO_A:g}$"
HIST_BINS = N_BINS
edges = observable_bin_edges(HIST_BINS, OBSERVABLE)
distributions = []
for eta in ETA_VALUES:
    nu_s, nu_sbi, nu_b, nu_ni = templates.anchors(max(eta, REFERENCE_RATIO_ETA_MIN), HIST_BINS)[0]
    root_eta = np.sqrt(eta)
    coherent = (eta-root_eta)*nu_s + root_eta*nu_sbi + (1-root_eta)*nu_b
    distributions.append((eta, coherent, nu_ni))

# Each panel has its own scale. Signed components remain visible on symlog.
with plt.style.context(hep.style.ATLAS), plt.rc_context({'font.size': 14}):
    ncols = min(2, len(distributions)); nrows = (len(distributions)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.8*nrows), squeeze=False)
    for ax, (eta, coherent, ni) in zip(axes.flat, distributions):
        total = coherent+ni
        if np.any(coherent < 0):
            hep.histplot([ni, coherent, total], bins=edges, histtype='step',
                         color=['#888888','#0072B2','black'],
                         label=['Non-interfering background', 'Coherent SBI', 'Total'], yerr=False, ax=ax)
            ax.set_yscale('symlog', linthresh=.1)
            ax.text(.04,.79,f'{np.count_nonzero(total < 0)} negative total-yield bins', transform=ax.transAxes, fontsize=11)
        else:
            hep.histplot([ni,coherent], bins=edges, stack=True, histtype='fill',
                         color=['#B8B8B8','#56B4E9'], edgecolor='black', linewidth=.7,
                         label=['Non-interfering background', r'Coherent SBI ($\mu=\eta$)'], yerr=False, ax=ax)
            ax.set_yscale('log')
            positive=total[total>0]
            if len(positive): ax.set_ylim(max(.001,positive.min()*.3),positive.max()*30)
        ax.text(.04,.94,fr'$\eta={eta:g},\quad\alpha_{{NI}}=0$',transform=ax.transAxes,va='top')
        ax.set(xlim=(0,1),xlabel=OBSERVABLE_LABEL,ylabel='Expected events / bin')
        ax.legend(fontsize=10,loc='upper right'); ax.grid(False)
    for ax in axes.flat[len(distributions):]: ax.set_visible(False)
    fig.tight_layout()
    for ext in ('png','pdf'): fig.savefig(run.path('plots', f'08_reference_ratio_distributions.{ext}'),dpi=160,bbox_inches='tight')
    plt.show()
display(pd.DataFrame([dict(eta=e, coherent_SBI_yield=c.sum(), NI_yield=n.sum(),
                           total_yield=(c+n).sum(),minimum_bin_yield=(c+n).min()) for e,c,n in distributions]))

In [ ]:
collapse = templates.collapse_table(CHECK_ETAS, N_BINS)
display(collapse)
assert collapse.loc[collapse.eta.eq(1.), 'occupied_bins'].item() == 1
np.testing.assert_array_equal(collapse.loc[collapse.eta.eq(1.), ['z_min','z_max']].to_numpy(), [[.5,.5]])
np.testing.assert_allclose(templates.anchors(1.,N_BINS)[0].sum(axis=-1), samples['rates'][:4], rtol=1e-12)
collapse.to_csv(run.path('results','reference_ratio_collapse.csv'),index=False)

### Freeze the observable inside each fit

At each test set $\eta=\mu_0$, then freeze learned event coordinates, edges and
MC Asimov observations throughout the null and free fits. Only the physical
coefficients and NI nuisance change. The no-floor setting does not restrict
the physical fit domain. Learned ratios carry no exact sufficiency guarantee.

### Refresh the analytical unbinned benchmark

This is the only oracle inference calculation. It provides a reference for the
learned-coordinate MC histograms. Notebook 03 retains its separate learned-model
closure comparisons. The new benchmark is saved in the practical evaluation.

In [ ]:
from poodemo.practical import refresh_analytic_reference
unbinned = refresh_analytic_reference(run)
display(unbinned)
print("Analytical unbinned benchmark refreshed; all histogram inputs remain simulation-based.")

### MC convergence at fixed map and yields

In [ ]:
mc_scans = convergence_scans(run, templates, etas=scan_grid, n_bins=N_BINS)
display(mc_scans.groupby(['selected_per_source','systematics']).agg(
    scan_points=('valid','size'), valid_points=('valid','sum')))
fig, axes = plt.subplots(1,2,figsize=(13,4.5),sharex=True)
for ax, syst in zip(axes,[False,True]):
    for size, group in mc_scans.loc[mc_scans.systematics.eq(syst)].groupby('selected_per_source'):
        group=group.sort_values('mu')
        line,=ax.plot(group.mu,group.q.where(group.valid),label=f'{size:,} selected / source')
        bad=group.loc[~group.valid]
        ax.plot(bad.mu,np.zeros(len(bad)),marker='x',ls='none',color=line.get_color(),
                transform=ax.get_xaxis_transform(),clip_on=False)
    ref=unbinned.loc[unbinned.systematics.eq(syst)].sort_values('mu')
    ax.plot(ref.mu,ref.q.where(ref.valid),'k--',label='Analytical unbinned')
    ax.set(xlabel=r'Tested $\mu_0$',ylabel=r'$q(\mu_0)$',title='NI profiled' if syst else 'NI fixed',ylim=(0,None))
    ax.legend(fontsize=8)
fig.suptitle('MC convergence: crosses on the axis mark invalid fits')
fig.tight_layout();fig.savefig(run.path('plots','08_mc_convergence.png'),dpi=160,bbox_inches='tight');plt.show()

In [ ]:
from poodemo.pipeline import run_binning_study
binning_run = run.with_config(bin_counts=BIN_COUNTS)
binning = run_binning_study(binning_run, observable="reference_ratio", template_bank=templates)
display(binning["scans"])
display(binning["fidelity"])
display(binning['scans'].groupby(['model','n_bins'],dropna=False).agg(
    scan_points=('valid','size'), valid_points=('valid','sum')))

In [ ]:
def plot_scans(frame, title, filename, group_columns=None, *, xlim=None, ylim=None):
    """Plot saved scans with optional (min, max) axis limits.

    Omit xlim/ylim to retain the default view. A None endpoint keeps that
    bound unchanged, for example ylim=(0, None) or xlim=(None, 1.2).
    """
    x_column = next((c for c in ["mu", "mu_test", "mu0"] if c in frame.columns), None)
    y_column = next((c for c in ["q", "t_mu", "test_statistic", "ts"] if c in frame.columns), None)
    if x_column is None or y_column is None:
        raise ValueError(f"Expected mu and q scan columns; found {list(frame.columns)}")
    if group_columns is None:
        group_columns = [c for c in ["method", "model", "systematics", "nuisances", "n_bins", "bins", "observable"] if c in frame.columns]
    groups = frame.groupby(group_columns, dropna=False, sort=False) if group_columns else [("scan", frame)]
    fig, ax = plt.subplots()
    for label, group in groups:
        group = group.sort_values(x_column)
        values = label if isinstance(label, tuple) else (label,)
        parts = []
        for column, value in zip(group_columns, values):
            if pd.isna(value):
                continue
            if column == "systematics":
                parts.append("profiled" if value else "stat only")
            elif column in ("n_bins", "bins"):
                parts.append(f"{int(value)} bins")
            else:
                parts.append(str(value))
        ax.plot(group[x_column], group[y_column].where(group["valid"]) if "valid" in group else group[y_column], marker=".", ms=3, label=" | ".join(parts))
        # Invalid fits are shown on the x axis, never joined by a valid curve.
        if "valid" in group:
            bad = group.loc[~group["valid"]]
            ax.plot(bad[x_column], np.zeros(len(bad)), "rx", transform=ax.get_xaxis_transform(), clip_on=False)
    ax.axhline(1.0, color="0.45", ls=":", lw=1)
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t(\mu_0)$", title=title)
    ax.set_ylim(bottom=0)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    fig.tight_layout()
    output = run.path("plots")
    output.mkdir(exist_ok=True)
    fig.savefig(output / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
SCAN_BIN_COUNTS = [4, 12, 24, 36, 60]
scans = binning["scans"]
shown_scans = scans.loc[
    (scans["model"].eq("direct reference_ratio histogram") & scans["n_bins"].isin(SCAN_BIN_COUNTS))
    | scans["model"].eq("analytic unbinned")
    | scans["model"].str.startswith("learned unbinned", na=False)
].copy()
SCAN_XLIM = None  # Set None to show the full scan range.
SCAN_YLIM = None  # Set None to show the full vertical range.
scan_title = "Ratio histograms: " + ", ".join(map(str, SCAN_BIN_COUNTS)) + " bins"
plot_scans(shown_scans, scan_title, "08_binning_scans.pdf",
           xlim=SCAN_XLIM, ylim=SCAN_YLIM);

### Finite-separation tests and local information

Learned ratios need not be sufficient. Compare the binned likelihood with the
analytical unbinned benchmark. Local information at $\mu=\eta$ differs from the
later width study at fixed truth $\mu_*=1$. MC template noise can put information
ratios above one; that is a convergence diagnostic, not a physical gain over the
full likelihood. At $\eta=1$ the histogram retains only rate information.

In [ ]:
PLOT_PREFIX = '08'

fidelity = binning["fidelity"]
display(fidelity)
eta_near_truth = min(fidelity["eta"].unique(), key=lambda e: abs(e - run.config["asimov_mu"]))
at_anchor = fidelity[np.isclose(fidelity["eta"], eta_near_truth)].sort_values("n_bins")
y_columns = [c for c in fidelity if "information_fraction" in c]
if y_columns:
    ax = at_anchor.plot(x="n_bins", y=y_columns, marker="o", logx=True)
    ax.set(xlabel="Number of bins", ylabel="Retained Fisher information fraction",
           title=fr"Information retained at $\eta={eta_near_truth:.3g}$")
    ax.figure.tight_layout()
    (run.path("plots")).mkdir(exist_ok=True)
    ax.figure.savefig(run.path("plots") / f"{PLOT_PREFIX}_binning_diagnostics.pdf", bbox_inches="tight")
    plt.show()

### Estimator and local uncertainty versus the frozen observable

Now hold the generating experiment fixed at $(\mu_*,\alpha_{NI})=(\mu_*,0)$
and vary only the observable anchor $\eta$. Each row refills the
**same Asimov expectation**, freezes that histogram, and fits physical
$\mu$, with NI either fixed or profiled. For an identified, correctly
modeled Asimov experiment, $\hat\mu(\eta)$ should remain at the
generating value. This is not a study of finite-sample fluctuations.

The width is evaluated from the local Fisher information at the fixed
truth, using the extended likelihood and the NI auxiliary constraint:
$$
\sigma_\mu^{\rm fixed}(\eta)=1/\sqrt{I_{\mu\mu}^{(\eta)}},\qquad
\sigma_\mu^{\rm profiled}(\eta)
=\sqrt{[(\mathbf I^{(\eta)})^{-1}]_{\mu\mu}}.
$$
This is the local Asimov curvature of **each frozen-observable fit**.
It is neither the curvature of the scan that changes $\eta=\mu_0$
at every point nor a global confidence interval. A second likelihood
minimum can produce additional accepted parameter regions that a local
width cannot describe.

The full unbinned reference does not depend on $\eta$. For the binned
comparison, profiled widths use nuisance morphing after binning, as in
the spline notebook; its difference from integrating the unbinned morphing is a
separate model effect. Inspect anomalous fitted values for optimizer
failure or multiple equally good minima before interpreting them as
compression bias.

Use the primary 36-bin estimator resolution, explicitly passed as `N_BINS`.
The independent `BIN_COUNTS` sweep does not change this estimator setting.
The truth stays fixed while $\eta$ changes. The baseline uses no floor. At $\eta=1$ this is a rate-only measurement: multiple physical $\mu$ values
can have equal rates. A different fitted branch need not indicate optimizer
failure. Local widths do not describe global ambiguity; invalid widths are
reported rather than floored.

The next cells compare **no floor** with an explicit **$0.2$ floor control**,
using the same 36 bins, fresh MC bank, Asimov truth, $\eta$ grid, nuisance treatments
and physical fit bounds. Primary estimator outputs retain the baseline settings;
the positive-floor control writes separate `reference_ratio_estimator_eta_floor_control`
files. A low-$\eta$ zoom highlights the affected region; the vertical line marks
the comparison floor.
These are **local widths at the fixed Asimov truth**, not confidence-interval
coverage or a proof of improved global likelihood fits. The free-fit MLE panels
remain a separate diagnostic. Values at $\eta\geq0.2$ should agree between
the two binned prescriptions because their partitions are then identical.

In [ ]:
from poodemo.diagnostics import run_estimator_study

print("Estimator-study bin count:", N_BINS)
# Primary study uses the configured baseline (no floor by default).
estimator_study = run_estimator_study(run, observable="reference_ratio", n_bins=N_BINS, template_bank=templates)
# Explicit floor control; it must remain distinct when the baseline is zero.
no_floor_run = run.with_config(reference_ratio_eta_floor=0.0)
estimator_no_floor = (estimator_study if REFERENCE_RATIO_ETA_MIN == 0.0 else
    run_estimator_study(no_floor_run, observable="reference_ratio", n_bins=N_BINS, template_bank=templates,
                        output_tag="reference_ratio_estimator_eta_no_floor"))
floor_run = run.with_config(reference_ratio_eta_floor=COMPARISON_ETA_MIN)
estimator_floor = (estimator_study if REFERENCE_RATIO_ETA_MIN == COMPARISON_ETA_MIN else
    run_estimator_study(floor_run, observable="reference_ratio", n_bins=N_BINS, template_bank=templates,
                        output_tag="reference_ratio_estimator_eta_floor_control"))
np.testing.assert_array_equal(np.sort(estimator_floor["eta"].unique()),
                              np.sort(estimator_no_floor["eta"].unique()))
FLOOR_LABEL = f"{N_BINS} bins, floor={COMPARISON_ETA_MIN:g}"
NO_FLOOR_LABEL = f"{N_BINS} bins, no floor"
# Include a single analytical baseline per eta and NI treatment.
is_unbinned = estimator_study["model"].eq("analytic unbinned")
estimator_comparison = pd.concat([
    estimator_floor.loc[~estimator_floor["model"].eq("analytic unbinned")]
        .assign(observable_variant=FLOOR_LABEL),
    estimator_no_floor.loc[~estimator_no_floor["model"].eq("analytic unbinned")]
        .assign(observable_variant=NO_FLOOR_LABEL),
    estimator_study.loc[is_unbinned].assign(observable_variant="Analytical unbinned"),
], ignore_index=True)
display(estimator_comparison)
invalid_rows = estimator_comparison[
    ~estimator_comparison["fit_valid"] | ~estimator_comparison["width_valid"]]
if len(invalid_rows):
    print("Invalid fits or local widths (omitted from the corresponding curves):")
    display(invalid_rows)
estimator_comparison.to_csv(
    run.path("results", "reference_ratio_estimator_eta_floor_comparison.csv"), index=False)
print("Primary baseline estimator table:", run.path("results", "reference_ratio_estimator_eta.csv"))
print("Floor control (when different from baseline):", run.path("results", "reference_ratio_estimator_eta_floor_control.csv"))

In [ ]:
PLOT_PREFIX = '08'
ESTIMATOR_ZOOM_XLIM = (0.0, 0.5)  # Low-eta width comparison; edit to change the zoom.

import mplhep as hep

truth_mu = float(estimator_study["truth"].iloc[0])
estimator_styles = {
    FLOOR_LABEL: dict(color="#D55E00", linestyle="--", linewidth=1.8),
    NO_FLOOR_LABEL: dict(color="#0072B2", linestyle="-", linewidth=2.2),
    "Analytical unbinned": dict(color="black", linestyle=":", linewidth=2.0),
}

def estimator_curves(systematic):
    subset = estimator_comparison.loc[estimator_comparison["systematics"].eq(systematic)]
    for label, style in estimator_styles.items():
        values = subset.loc[subset["observable_variant"].eq(label)].sort_values("eta")
        yield label, values, style

def mark_estimator_floor(ax):
    if COMPARISON_ETA_MIN > 0:
        ax.axvline(COMPARISON_ETA_MIN, color="0.55", ls="--", lw=1,
                   label=fr"Control floor $\eta={COMPARISON_ETA_MIN:g}$")
    ax.grid(False)

with plt.style.context(hep.style.ATLAS), plt.rc_context({"font.size": 14}):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, squeeze=False)
    for row, systematic in enumerate([False, True]):
        for label, values, style in estimator_curves(systematic):
            sigma = values["sigma_mu"].replace([np.inf, -np.inf], np.nan).where(values["width_valid"])
            axes[row, 0].plot(values["eta"], sigma, label=label, **style)
            axes[row, 1].plot(values["eta"], values["mu_hat"].where(values["fit_valid"]),
                              label=label, **style)
        nuisance_label = "NI profiled" if systematic else "NI fixed at zero"
        axes[row, 0].set(title=nuisance_label, ylabel=r"Local width $\sigma_\mu(\eta)$")
        axes[row, 1].axhline(truth_mu, color="0.65", ls="-.", lw=1, label="Generating value")
        axes[row, 1].set(title=nuisance_label + ": same Asimov experiment", ylabel=r"$\hat\mu(\eta)$")
        subset = estimator_comparison.loc[estimator_comparison["systematics"].eq(systematic)]
        finite_estimates = subset.loc[subset["fit_valid"], "mu_hat"].to_numpy()
        finite_estimates = finite_estimates[np.isfinite(finite_estimates)]
        lower = min(truth_mu, finite_estimates.min()) if len(finite_estimates) else truth_mu
        upper = max(truth_mu, finite_estimates.max()) if len(finite_estimates) else truth_mu
        margin = max(.03 * (run.config["mu_max"] - run.config["mu_min"]), .1 * (upper - lower))
        axes[row, 1].set_ylim(lower - margin, upper + margin)
        for ax in axes[row]:
            mark_estimator_floor(ax)
            ax.set_xlabel(r"Frozen observable anchor $\eta$")
            ax.legend(fontsize=8, loc="best")
    fig.tight_layout()
    (run.path("plots")).mkdir(exist_ok=True)
    for extension in ("pdf", "png"):
        fig.savefig(run.path("plots") / f"{PLOT_PREFIX}_estimator_eta.{extension}",
                    bbox_inches="tight", dpi=160)
    plt.show()

    # Same values, with vertical limits set from the displayed low-eta region.
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True, squeeze=False)
    for ax, systematic in zip(axes.flat, [False, True]):
        shown_widths = []
        for label, values, style in estimator_curves(systematic):
            sigma = values["sigma_mu"].replace([np.inf, -np.inf], np.nan).where(values["width_valid"])
            ax.plot(values["eta"], sigma, label=label, **style)
            in_view = values["eta"].between(*ESTIMATOR_ZOOM_XLIM)
            shown_widths.extend(sigma.loc[in_view & sigma.notna()].tolist())
        if shown_widths:
            lo, hi = min(shown_widths), max(shown_widths)
            pad = max(.08 * (hi - lo), .02 * max(abs(lo), abs(hi)), 1e-8)
            ax.set_ylim(max(0., lo - pad), hi + pad)
        mark_estimator_floor(ax)
        ax.set(xlim=ESTIMATOR_ZOOM_XLIM, xlabel=r"Frozen observable anchor $\eta$",
               ylabel=r"Local width $\sigma_\mu(\eta)$",
               title="NI profiled" if systematic else "NI fixed at zero")
        ax.legend(fontsize=9, loc="best")
    fig.suptitle(fr"Same Asimov truth $\mu_*={truth_mu:g}$: floor versus no floor", fontsize=16)
    fig.tight_layout()
    for extension in ("pdf", "png"):
        fig.savefig(run.path("plots") / f"{PLOT_PREFIX}_estimator_eta_floor_comparison_zoom.{extension}",
                    bbox_inches="tight", dpi=160)
    plt.show()

Review the MC convergence and validity diagnostics before updating 09.
This change updates **08 only**. Notebook 09 still uses its earlier bank;
it does not yet consume this 10M Colab cache. The cached selected coordinates
and learned ratios are retained for a subsequent interpolation study.